# SBE26 Data Processing

Processing path used:

build_sbe26_dataset_from_metadata -> IMOSNetCDFConverter_SBE26 (internal proc_1 filename)

### Setup

Imports

In [ ]:
import os
import sys
import importlib
from pathlib import Path

import numpy as np
import pandas as pd

Import local tools

In [ ]:
TOOLS_DIR = Path.cwd().resolve().parent
if str(TOOLS_DIR) not in sys.path:
    sys.path.insert(0, str(TOOLS_DIR))

from tools.imos_nc_converter import imos_converter as imos_converter_module
importlib.reload(imos_converter_module)
IMOSNetCDFConverter_SBE26 = imos_converter_module.IMOSNetCDFConverter_SBE26

from tools import database_lookup as database_lookup_module
importlib.reload(database_lookup_module)
get_instrument_context = database_lookup_module.get_instrument_context
update_metadata_file_fields = database_lookup_module.update_metadata_file_fields

from tools.parsers import read_sbe26 as read_sbe26_module
importlib.reload(read_sbe26_module)
build_sbe26_dataset_from_metadata = read_sbe26_module.build_sbe26_dataset_from_metadata

from tools.helpers import plot_data_by_qc

Definitions

In [ ]:
# Working directory
os.chdir("/datasets/work/oa-srsalt/work/preqa/SWOT/cal_val/jason_calval/all_mooring_data/ash")

In [ ]:
# Select instrument using ID from satellite_altimetry_moorings_metadata.csv

inst_deploy_id = 259    # update per deployment

database, _row, cfg, metadata = get_instrument_context(
    inst_deploy_id=inst_deploy_id,
    print_details=True,
)

In [ ]:
converter = IMOSNetCDFConverter_SBE26(input_folder="", input_file="", output_dir="")

Read and create ds, df

In [ ]:
# SBE26 averaging parameters
pressure_threshold = 37.0
window_seconds = 4 * 60
output_freq = "5min"

In [ ]:
raw_df, averaged_df, ds_proc1, input_file, proc_info = build_sbe26_dataset_from_metadata(
    _row,
    cwd=Path.cwd(),
    pressure_threshold=pressure_threshold,
    window_seconds=window_seconds,
    output_freq=output_freq,
    verbose=True,
)

print(f"Raw rows: {len(raw_df)}")
print(f"Averaged rows: {len(averaged_df)}")
print(f"Input file used: {input_file}")

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Quick interactive plot of raw data before pressure-threshold filtering/chopping
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    subplot_titles=("Pressure [dbar]", "Temperature [degC]"),
)

fig.add_trace(
    go.Scatter(
        x=raw_df.index,
        y=raw_df["Pressure [db]"],
        mode="lines",
        name="Pressure [db]",
        line=dict(width=1),
    ),
    row=1,
    col=1,
)

fig.add_hline(
    y=pressure_threshold,
    line_dash="dash",
    line_color="red",
    annotation_text=f"threshold={pressure_threshold}",
    annotation_position="bottom right",
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=raw_df.index,
        y=raw_df["TEMP"],
        mode="lines",
        name="TEMP",
        line=dict(width=1, color="orange"),
    ),
    row=2,
    col=1,
)

fig.update_layout(
    title="Raw SBE26 data (pre-threshold)",
    height=650,
    width=1150,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)

fig.update_xaxes(title_text="Time", row=2, col=1)
fig.update_yaxes(title_text="Pressure [dbar]", row=1, col=1)
fig.update_yaxes(title_text="Temperature [degC]", row=2, col=1)

fig.show()

In [ ]:
file_variables = list(ds_proc1.variables)
print("file variables:", file_variables)

In [ ]:
PLOT_VARS = None  # set None for defaults, to specify use ["PRES", "TEMP"]
fig = plot_data_by_qc(
    ds_proc1,
    variables=PLOT_VARS,
    flags_to_plot=[1],      # optional
    y_zoom_to_good=True,    # optional
)
fig.show()

In [ ]:
time_coverage_start = proc_info["time_coverage_start"]
time_coverage_end = proc_info["time_coverage_end"]
deploy_start = _row.get("deploy_date", time_coverage_start)
deploy_end = _row.get("recovery_date", time_coverage_end)
inst_channels = str(_row.get("mooring_channels", "PT"))
converter_depth = float(_row.get("nominal_depth", 0.0))

print(f"time_coverage_start: {time_coverage_start}")
print(f"time_coverage_end: {time_coverage_end}")

Write trimmed file to proc_1

In [ ]:
proc_1_source_path = str((Path.cwd() / "proc_1_source_sbe26.nc").resolve())
ds_proc1.to_netcdf(proc_1_source_path)

proc_1_out = converter.process(
    input_nc_path=proc_1_source_path,
    longitude=float(_row["longitude"]),
    latitude=float(_row["latitude"]),
    depth=converter_depth,
    inst_channels=inst_channels,
    start_of_good_data=time_coverage_start,
    time_deployment_start=deploy_start,
    time_deployment_end=deploy_end,
    site_code=str(_row["location"]),
    version=str(_row.get("version", "1")),
    instrument=str(_row["inst_type"]),
    inst_id=str(int(_row["inst_id"])),
    location=str(_row["location"]),
    output_name_mode="internal",
    output_stage="proc_1",
    metadata_row=_row,
    metadata_mode="fill_missing",
    deployment_id=_row.get("deployment_id", ""),
    mooring_channels=_row.get("mooring_channels", ""),
    nominal_inst_depth=_row.get("nominal_inst_depth", ""),
    deploy_date=_row.get("deploy_date", ""),
    recovery_date=_row.get("recovery_date", ""),
    time_coverage_start=time_coverage_start,
    time_coverage_end=time_coverage_end,
)

print(f"proc_1 output: {proc_1_out}")
Path(proc_1_source_path).unlink(missing_ok=True)

Write proc_1 filename to metadata table

In [ ]:
proc_1_name = Path(proc_1_out).name
_row = update_metadata_file_fields(
    inst_deploy_id,
    {"proc_1_file": proc_1_name},
    output_paths={"proc_1_file": proc_1_out},
    working_dir=Path.cwd(),
)
print(f"Updated proc_1_file: {_row['proc_1_file']}")